# Airline Operations & Disruption Intelligence
## Weather Data Collection & Cleaning

Collect historical hourly weather data for airports used in the cleaned BTS flight dataset.

**Period:** January 1, 2026 to March 31, 2026

**Weather variables:** temperature, precipitation, wind speed, weather code

## 1. Import libraries

In [22]:
import pandas as pd
import numpy as np
import requests
import json
import time
from pathlib import Path

## 2. Define project paths

In [23]:
project_path = Path(r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence")

flights_path = r"D:\Data Analyst\EXCEL\api\processed\flights_2026_q1_cleaned.csv"
airport_path = r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\raw\airports\airport_reference.csv"

weather_raw_path = project_path / "data" / "raw" / "weather"
weather_processed_path = project_path / "data" / "processed"

weather_raw_path.mkdir(parents=True, exist_ok=True)
weather_processed_path.mkdir(parents=True, exist_ok=True)

print("Project path:", project_path)

Project path: D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence


## 3. Load cleaned flight data

In [24]:
flights = pd.read_csv(flights_path)
print('Rows:', len(flights))
print('Columns:', len(flights.columns))
flights.head()

Rows: 1847242
Columns: 55


,year,quarter,month,day_of_month,day_of_week,fl_date,mkt_unique_carrier,branded_code_share,mkt_carrier_airline_id,mkt_carrier,...,missing_operational_duration,extreme_dep_delay,extreme_arr_delay,is_cancelled,is_diverted,is_irregular_operation,is_departure_delayed,is_arrival_delayed,severe_departure_delay,severe_arrival_delay
0,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,False,False,False,0,0,0,0,1,0,0
1,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,False,False,False,0,0,0,0,0,0,0
2,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,False,False,False,0,0,0,0,0,0,0
3,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,False,False,False,0,0,0,0,0,0,0
4,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,False,False,False,0,0,0,0,0,0,0


## 4. Check columns needed for weather integration

In [25]:
required_columns = ['origin', 'fl_date', 'crs_dep_time']
for column in required_columns:
    if column in flights.columns:
        print(column, '✓')
    else:
        print(column, 'MISSING')

origin ✓
fl_date ✓
crs_dep_time ✓


## 5. Load airport reference data

In [26]:
airport_ref = pd.read_csv(airport_path)
print('Rows:', len(airport_ref))
print('Columns:', len(airport_ref.columns))
airport_ref.head()

Rows: 362
Columns: 7


,iata_code,airport_name,city,country,latitude,longitude,timezone
0,PPG,Pago Pago International Airport,Pago Pago,American Samoa,-14.331000,-170.710007,-11
1,SPN,Saipan International Airport,Saipan,Northern Mariana Islands,15.119000,145.729004,10
2,GUM,Antonio B. Won Pat International Airport,Agana,Guam,13.483400,144.796005,10
3,STT,Cyril E. King Airport,St. Thomas,Virgin Islands,18.337299,-64.973396,-4
4,STX,Henry E Rohlsen Airport,St. Croix Island,Virgin Islands,17.701900,-64.798599,-4


## 6. Keep only columns needed for weather

In [27]:
airport_weather_ref = airport_ref[['iata_code', 'latitude', 'longitude']].copy()
airport_weather_ref.head()

,iata_code,latitude,longitude
0,PPG,-14.331000,-170.710007
1,SPN,15.119000,145.729004
2,GUM,13.483400,144.796005
3,STT,18.337299,-64.973396
4,STX,17.701900,-64.798599


## 7. Validate airport reference

In [28]:
print('Total airport records:', len(airport_weather_ref))
print('Unique airport codes:', airport_weather_ref['iata_code'].nunique())
print('\nMissing values:')
print(airport_weather_ref.isna().sum())
print('\nDuplicate airport codes:', airport_weather_ref['iata_code'].duplicated().sum())

Total airport records: 362
Unique airport codes: 362

Missing values:
iata_code    0
latitude     0
longitude    0
dtype: int64

Duplicate airport codes: 0


## 8. Identify airports used by flights

In [29]:
flight_airports = flights['origin'].dropna().astype(str).str.strip().unique()
flight_airports_set = set(flight_airports)
print('Unique ORIGIN airports:', len(flight_airports))

Unique ORIGIN airports: 362


## 9. Check airport reference coverage

In [30]:
reference_airports_set = set(airport_weather_ref['iata_code'])
missing_airports = flight_airports_set - reference_airports_set
print('Airports missing from reference:', len(missing_airports))
print(missing_airports)

Airports missing from reference: 0
set()


Do not silently remove unmatched airports. Investigate and verify them first. XWA and EAR were previously verified and added to the reference table.

## 10. Create final airport list

In [31]:
weather_airports = airport_weather_ref[airport_weather_ref['iata_code'].isin(flight_airports_set)].copy()
print('Airports requiring weather data:', len(weather_airports))
weather_airports.head(20)

Airports requiring weather data: 362


,iata_code,latitude,longitude
0,PPG,-14.331000,-170.710007
1,SPN,15.119000,145.729004
2,GUM,13.483400,144.796005
3,STT,18.337299,-64.973396
4,STX,17.701900,-64.798599
5,BQN,18.494900,-67.129402
6,PSE,18.008301,-66.563004
7,SJU,18.439400,-66.001801
8,ITO,19.721399,-155.048004
9,FSM,35.336601,-94.367401


## 11. Validate coordinates

In [32]:
print('Missing latitude:', weather_airports['latitude'].isna().sum())
print('Missing longitude:', weather_airports['longitude'].isna().sum())
print('Duplicate airport codes:', weather_airports['iata_code'].duplicated().sum())

Missing latitude: 0
Missing longitude: 0
Duplicate airport codes: 0


## 12. Open-Meteo API settings

We make one request per airport for the full 90-day period, not one request per flight. The API response is requested in UTC; local-time handling will be addressed carefully during the later flight-weather join.

In [33]:
weather_url = 'https://archive-api.open-meteo.com/v1/archive'
start_date = '2026-01-01'
end_date = '2026-03-31'
weather_variables = ['temperature_2m', 'precipitation', 'wind_speed_10m', 'weather_code']
print(weather_url, start_date, end_date)

https://archive-api.open-meteo.com/v1/archive 2026-01-01 2026-03-31


## 13. Test the API with ATL

In [34]:
atl = weather_airports[weather_airports['iata_code'] == 'ATL'].iloc[0]
atl_latitude = atl['latitude']
atl_longitude = atl['longitude']
print('Airport: ATL')
print('Latitude:', atl_latitude)
print('Longitude:', atl_longitude)

Airport: ATL
Latitude: 33.6367
Longitude: -84.428101


In [35]:
params = {
    'latitude': atl_latitude,
    'longitude': atl_longitude,
    'start_date': start_date,
    'end_date': end_date,
    'hourly': ','.join(weather_variables),
    'timezone': 'UTC'
}
response = requests.get(weather_url, params=params, timeout=60)
print('API Status Code:', response.status_code)

ConnectionError: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=33.6367&longitude=-84.428101&start_date=2026-01-01&end_date=2026-03-31&hourly=temperature_2m%2Cprecipitation%2Cwind_speed_10m%2Cweather_code&timezone=UTC (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000017AE43B8910>: Failed to resolve 'archive-api.open-meteo.com' ([Errno 11001] getaddrinfo failed)"))

In [36]:
if response.status_code == 200:
    print('API request successful.')
else:
    print('API request failed.')
    print(response.text)

NameError: name 'response' is not defined

In [ ]:
data = response.json()
print(data.keys())
if 'hourly' in data:
    print(data['hourly'].keys())
    print(data['hourly']['time'][:5])
else:
    print(data)

## 14. Save the ATL raw response

In [ ]:
atl_raw_file = weather_raw_path / 'ATL_2026-01-01_to_2026-03-31.json'
with open(atl_raw_file, 'w', encoding='utf-8') as file:
    json.dump(data, file, indent=4)
print('Raw API response saved:', atl_raw_file)

## 15. Convert ATL response into a table

In [ ]:
hourly = data['hourly']
atl_weather = pd.DataFrame({
    'time': hourly['time'],
    'temperature': hourly['temperature_2m'],
    'precipitation': hourly['precipitation'],
    'wind_speed': hourly['wind_speed_10m'],
    'weather_code': hourly['weather_code']
})
atl_weather['time'] = pd.to_datetime(atl_weather['time'])
atl_weather['date'] = atl_weather['time'].dt.date
atl_weather['hour'] = atl_weather['time'].dt.hour
atl_weather['airport_code'] = 'ATL'
atl_weather = atl_weather[['airport_code','date','hour','temperature','precipitation','wind_speed','weather_code']]
atl_weather.head()

## 16. Validate ATL test data

In [ ]:
print('Total rows:', len(atl_weather))
print('Duplicate records:', atl_weather.duplicated(subset=['airport_code','date','hour']).sum())
print('\nMissing values:')
print(atl_weather.isna().sum())
print('\nNegative precipitation:', (atl_weather['precipitation'] < 0).sum())
print('Negative wind speed:', (atl_weather['wind_speed'] < 0).sum())

In [ ]:
atl_weather_file = weather_processed_path / 'weather_ATL_2026_Q1.csv'
atl_weather.to_csv(atl_weather_file, index=False)
print('ATL test weather saved:', atl_weather_file)

## 17. Collect weather for all airports

The code below uses an existing raw JSON file when available. This prevents repeated API requests if the notebook is restarted. New requests are followed by a short pause.

In [ ]:
weather_results = []
failed_airports = []

In [ ]:
for index, row in weather_airports.iterrows():
    airport_code = row['iata_code']
    latitude = row['latitude']
    longitude = row['longitude']

    print('\n' + '=' * 50)
    print(f'Processing {airport_code} ({index + 1}/{len(weather_airports)})')

    raw_file = weather_raw_path / f'{airport_code}_2026-01-01_to_2026-03-31.json'

    if raw_file.exists():
        print('Raw file already exists. Using saved file.')
        try: 
            with open(raw_file, 'r', encoding='utf-8') as file:
                data = json.load(file)
        except Exception as error:
            print('Could not read saved file:', error)
            failed_airports.append({'airport_code': airport_code, 'reason': 'Could not read saved JSON'})
            continue
    else:
        params = {
            'latitude': latitude,
            'longitude': longitude,
            'start_date': start_date,
            'end_date': end_date,
            'hourly': ','.join(weather_variables),
            'timezone': 'UTC'
        }
        try:
            response = requests.get(weather_url, params=params, timeout=60)
            print('API Status Code:', response.status_code)
        except requests.RequestException as error:
            print('API request failed:', error)
            failed_airports.append({'airport_code': airport_code, 'reason': str(error)})
            continue

        if response.status_code != 200:
            print('API failed.')
            failed_airports.append({'airport_code': airport_code, 'reason': response.text})
            continue

        data = response.json()
        if 'hourly' not in data:
            print('Hourly weather data missing.')
            failed_airports.append({'airport_code': airport_code, 'reason': 'hourly data missing'})
            continue

        with open(raw_file, 'w', encoding='utf-8') as file:
            json.dump(data, file, indent=4)
        print('Raw response saved.')
        time.sleep(1)

    hourly = data['hourly']
    weather_df = pd.DataFrame({
        'time': hourly['time'],
        'temperature': hourly['temperature_2m'],
        'precipitation': hourly['precipitation'],
        'wind_speed': hourly['wind_speed_10m'],
        'weather_code': hourly['weather_code']
    })
    weather_df['time'] = pd.to_datetime(weather_df['time'])
    weather_df['date'] = weather_df['time'].dt.date
    weather_df['hour'] = weather_df['time'].dt.hour
    weather_df['airport_code'] = airport_code
    weather_df = weather_df[['airport_code','date','hour','temperature','precipitation','wind_speed','weather_code']]
    weather_results.append(weather_df)
    print('Rows collected:', len(weather_df))

## 18. Combine all airports into ONE DataFrame

In [ ]:
weather = pd.concat(weather_results, ignore_index=True)
print('Total weather rows:', len(weather))

## 19. Validate the combined weather data

In [ ]:
print('Shape:', weather.shape)
print('\nFirst 5 rows:')
display(weather.head())
print('\nLast 5 rows:')
display(weather.tail())

In [ ]:
print('Required airports:', len(weather_airports))
print('Weather airports:', weather['airport_code'].nunique())

weather_airport_counts = weather['airport_code'].value_counts().sort_index()
print(weather_airport_counts.head())

In [ ]:
wrong_airport_counts = weather_airport_counts[weather_airport_counts != 2160]
print('Airports with unexpected row counts:', len(wrong_airport_counts))
if len(wrong_airport_counts) > 0:
    display(wrong_airport_counts)
else:
    print('All airports have 2,160 hourly records.')

In [ ]:
print('Start date:', weather['date'].min())
print('End date:', weather['date'].max())
print('\nMissing values:')
print(weather.isna().sum())

In [ ]:
duplicate_count = weather.duplicated(subset=['airport_code','date','hour']).sum()
print('Duplicate weather records:', duplicate_count)
print('Negative precipitation:', (weather['precipitation'] < 0).sum())
print('Negative wind speed:', (weather['wind_speed'] < 0).sum())

In [ ]:
print(weather['weather_code'].value_counts().sort_index())

## 20. Check failed airports

In [ ]:
failed_df = pd.DataFrame(failed_airports)
print('Failed airports:', len(failed_df))
if len(failed_df) > 0:
    display(failed_df)

In [ ]:
failed_file = weather_processed_path / 'weather_failed_airports.csv'
failed_df.to_csv(failed_file, index=False)
print('Failed airport log saved:', failed_file)

## 21. Save ONE combined weather CSV

In [ ]:
weather = weather[['airport_code','date','hour','temperature','precipitation','wind_speed','weather_code']]

weather_file = weather_processed_path / 'weather_cleaned.csv'
weather.to_csv(weather_file, index=False)

print('Final weather dataset saved:')
print(weather_file)

## 22. Final validation report

For 362 airports covering 90 days at 24 observations per day:

**362 × 90 × 24 = 781,920 rows**

The exact row count should be 781,920 if every airport has a complete hourly series.

In [ ]:
print('=' * 60)
print('FINAL WEATHER DATA VALIDATION')
print('=' * 60)
print('Total rows:', len(weather))
print('Unique airports:', weather['airport_code'].nunique())
print('Start date:', weather['date'].min())
print('End date:', weather['date'].max())
print('Duplicate records:', weather.duplicated(subset=['airport_code','date','hour']).sum())
print('\nMissing values:')
print(weather.isna().sum())
print('\nNegative precipitation:', (weather['precipitation'] < 0).sum())
print('Negative wind speed:', (weather['wind_speed'] < 0).sum())
print('\nFailed airports:', len(failed_df))
print('=' * 60)

# Weather collection completed

The cleaned weather dataset is now stored as **one CSV file**:

`data/processed/weather_cleaned.csv`

Raw API responses remain in:

`data/raw/weather/`

### Do not start weather-vs-delay analysis yet.

The next stage is the careful **flight → airport → weather integration**, including conversion of BTS `CRS_DEP_TIME` and handling the UTC/local-time issue before the merge.